# VectorFFT Benchmark Plots
VkFFT-style scatter plots for performance (GFLOP/s) and accuracy (roundtrip error).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    'figure.figsize': (14, 7),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 120,
})

In [ ]:
# Load data
import os

candidates = [
    '.',
    'csv',
    '../csv',
    '../../csv',
    r'C:\Users\Tugbars\Desktop\highSpeedFFT\src\stride-fft\csv',
    r'C:\Users\Tugbars\Desktop\highSpeedFFT\build\bin',
]
base = None
for d in candidates:
    p = os.path.join(d, 'vfft_perf_1d.csv')
    if os.path.exists(p):
        base = d
        break

if base is None:
    print(f"CWD: {os.getcwd()}")
    print("Searched:", candidates)
    raise FileNotFoundError("Can't find vfft_perf_1d.csv")

print(f'Loading from: {os.path.abspath(base)}')
perf = pd.read_csv(os.path.join(base, 'vfft_perf_1d.csv'))
acc  = pd.read_csv(os.path.join(base, 'vfft_acc_1d.csv'))
print(f'Performance: {len(perf)} rows, Accuracy: {len(acc)} rows')
perf.head()

In [ ]:
# ═══════════════════════════════════════════════════════════
# PERFORMANCE PLOT — VkFFT style
#
# VectorFFT: always BLUE, different marker shapes per category
# MKL:       always RED,  different marker shapes per category
# ═══════════════════════════════════════════════════════════

cat_markers = {
    'small':      {'marker': 'o', 'label': 'pow2 small (8-128)'},
    'pow2':       {'marker': 's', 'label': 'pow2 (256-131072)'},
    'composite':  {'marker': 'D', 'label': 'composite'},
    'prime_pow':  {'marker': '^', 'label': 'prime powers (3,5,7)'},
    'genfft':     {'marker': 'v', 'label': 'genfft (R=11,13)'},
    'rader':      {'marker': 'P', 'label': 'Rader primes'},
    'odd_comp':   {'marker': 'X', 'label': 'odd composites'},
    'mixed_deep': {'marker': '*', 'label': 'mixed deep'},
}

VFFT_COLOR = '#1f77b4'  # blue
MKL_COLOR  = '#d62728'  # red

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for idx, K in enumerate([4, 32, 256]):
    ax = axes[idx]
    sub = perf[perf['K'] == K].copy()
    
    for cat, style in cat_markers.items():
        d = sub[sub['category'] == cat]
        if len(d) == 0:
            continue
        
        # VectorFFT — blue filled
        ax.scatter(d['N'], d['vfft_gflops'],
                   c=VFFT_COLOR, marker=style['marker'],
                   s=40, alpha=0.85, edgecolors='none',
                   label=f"VFFT {style['label']}" if idx == 0 else None,
                   zorder=3)
        
        # MKL — red filled
        if 'mkl_gflops' in d.columns and d['mkl_gflops'].sum() > 0:
            ax.scatter(d['N'], d['mkl_gflops'],
                       c=MKL_COLOR, marker=style['marker'],
                       s=40, alpha=0.6, edgecolors='none',
                       label=f"MKL {style['label']}" if idx == 0 else None,
                       zorder=2)
    
    ax.set_xscale('log')
    ax.set_xlabel('FFT length')
    ax.set_ylabel('GFLOP/s' if idx == 0 else '')
    ax.set_title(f'FP64 batched 1D C2C — K={K}', fontweight='bold')
    ax.set_ylim(bottom=0)

# Legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4,
           bbox_to_anchor=(0.5, 1.02), fontsize=9, framealpha=0.9)

fig.suptitle('VectorFFT vs Intel MKL — 1D FFT Throughput', fontsize=14,
             fontweight='bold', y=1.08)
plt.tight_layout()
plt.savefig('vfft_throughput_1d.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: vfft_throughput_1d.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SPEEDUP PLOT — ratio vs MKL across all sizes and K values
# ═══════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 6))

for cat, style in cat_markers.items():
    d = perf[perf['category'] == cat]
    if len(d) == 0:
        continue
    
    for K, marker_size in [(4, 20), (32, 40), (256, 70)]:
        dk = d[d['K'] == K]
        if len(dk) == 0:
            continue
        ax.scatter(dk['N'], dk['ratio_vs_mkl'],
                   c=VFFT_COLOR, marker=style['marker'],
                   s=marker_size, alpha=0.8, edgecolors='none',
                   zorder=3)

ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)

ax.set_xscale('log')
ax.set_xlabel('FFT length')
ax.set_ylabel('Speedup vs MKL (higher = faster)')
ax.set_title('VectorFFT / MKL Speedup Ratio — All Categories', fontweight='bold')
ax.set_ylim(bottom=0)

# Legend: shapes for category, sizes for K
from matplotlib.lines import Line2D
cat_legend = [Line2D([0],[0], marker=s['marker'], color=VFFT_COLOR,
                     markersize=7, linestyle='None', label=s['label'])
              for s in cat_markers.values()]
size_legend = [
    Line2D([0],[0], marker='o', color='gray', markersize=4, linestyle='None', label='K=4'),
    Line2D([0],[0], marker='o', color='gray', markersize=6, linestyle='None', label='K=32'),
    Line2D([0],[0], marker='o', color='gray', markersize=9, linestyle='None', label='K=256'),
    Line2D([0],[0], color='black', linestyle='--', label='MKL parity'),
]
ax.legend(handles=cat_legend + size_legend, loc='upper right', fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('vfft_speedup_vs_mkl.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: vfft_speedup_vs_mkl.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ACCURACY PLOT — VkFFT style
#
# X: log(FFT length)
# Y: roundtrip error (log scale)
# Color: category
# Shows error growth with N (should be ~O(log N) * eps)
# ═══════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 6))

for cat, style in cat_style.items():
    d = acc[acc['category'] == cat]
    if len(d) == 0:
        continue
    ax.scatter(d['N'], d['roundtrip_err'],
               c=style['color'], marker=style['marker'],
               s=60, alpha=0.85, edgecolors='none',
               label=style['label'], zorder=3)

# Theoretical bound: O(log2(N) * eps)
eps = 2.22e-16
Ns = np.logspace(np.log10(8), np.log10(1e6), 200)
theoretical = np.log2(Ns) * eps * 5  # empirical constant
ax.plot(Ns, theoretical, 'k--', alpha=0.5, linewidth=1.5,
        label=r'$\sim \log_2(N) \cdot \varepsilon$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('FFT length')
ax.set_ylabel('Roundtrip error (max absolute)')
ax.set_title('VectorFFT FP64 Precision — Roundtrip Error vs FFT Length',
             fontweight='bold')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('vfft_precision.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: vfft_precision.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# COMBINED DENSE SCATTER — all K values overlaid (VkFFT style)
#
# VectorFFT = blue (different shapes per category)
# MKL = red (same shapes, dimmer)
# ═══════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(16, 8))

# Plot MKL first (background, red)
for cat, style in cat_markers.items():
    d = perf[perf['category'] == cat]
    if len(d) == 0 or d['mkl_gflops'].sum() == 0:
        continue
    ax.scatter(d['N'], d['mkl_gflops'],
               c=MKL_COLOR, marker=style['marker'],
               s=25, alpha=0.35, edgecolors='none',
               zorder=1)

# Plot VectorFFT on top (blue, vibrant)
for cat, style in cat_markers.items():
    d = perf[perf['category'] == cat]
    if len(d) == 0:
        continue
    ax.scatter(d['N'], d['vfft_gflops'],
               c=VFFT_COLOR, marker=style['marker'],
               s=35, alpha=0.85, edgecolors='none',
               label=style['label'], zorder=3)

# Library legend entries
from matplotlib.lines import Line2D
lib_legend = [
    Line2D([0],[0], marker='o', color=VFFT_COLOR, markersize=8, linestyle='None', label='VectorFFT'),
    Line2D([0],[0], marker='o', color=MKL_COLOR, markersize=8, linestyle='None', alpha=0.5, label='Intel MKL'),
]
cat_legend = [Line2D([0],[0], marker=s['marker'], color='gray',
                     markersize=7, linestyle='None', label=s['label'])
              for s in cat_markers.values()]
ax.legend(handles=lib_legend + cat_legend, loc='upper right', fontsize=9, ncol=2, framealpha=0.9)

ax.set_xscale('log')
ax.set_xlabel('FFT length', fontsize=12)
ax.set_ylabel('GFLOP/s', fontsize=12)
ax.set_title('VectorFFT FP64 Batched 1D C2C — All Sizes & Batch Counts',
             fontsize=14, fontweight='bold')
ax.set_ylim(bottom=0)

# Annotate peak
peak_row = perf.loc[perf['vfft_gflops'].idxmax()]
ax.annotate(f"Peak: {peak_row['vfft_gflops']:.0f} GF/s\nN={int(peak_row['N'])}, K={int(peak_row['K'])}",
            xy=(peak_row['N'], peak_row['vfft_gflops']),
            xytext=(peak_row['N']*3, peak_row['vfft_gflops']*0.9),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('vfft_scatter_all.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: vfft_scatter_all.png')

In [ ]:
# Summary stats
print('=== Summary ===')
print(f'Total benchmarks: {len(perf)}')
print(f'Win rate vs MKL:  {(perf["ratio_vs_mkl"] > 1.0).sum()}/{len(perf)}')
print(f'Min ratio:        {perf["ratio_vs_mkl"].min():.2f}x ({perf.loc[perf["ratio_vs_mkl"].idxmin(), "N"]})')
print(f'Max ratio:        {perf["ratio_vs_mkl"].max():.2f}x ({perf.loc[perf["ratio_vs_mkl"].idxmax(), "N"]})')
print(f'Median ratio:     {perf["ratio_vs_mkl"].median():.2f}x')
print()
print('Per category:')
summary = perf.groupby('category')['ratio_vs_mkl'].agg(['min','median','max','count'])
print(summary.to_string())